In [1]:
!pip -q install gradio

import gradio as gr

questions = [
    "👋 Welcome to the HR Interview! Tell me about yourself.",
    "Why do you want to work for our company?",
    "What are your strengths?",
    "What are your weaknesses?",
    "Where do you see yourself in 5 years?",
    "Why should we hire you?",
    "Tell me about a challenge you faced and how you solved it.",
    "How do you handle pressure or stress?",
    "What motivates you to do your best work?",
    "Do you have any questions for us?"
]

def respond(message, history, score):
    if not message.strip():
        return history, score

    history = history or []
    history.append({"role": "user", "content": message})

    # Scoring Logic: Give 1 point if the answer contains more than 5 words
    word_count = len(message.split())
    if word_count > 5:
        score += 1

    question_number = len([
        item for item in history if item["role"] == "user"
    ])

    if question_number < len(questions):
        reply = questions[question_number]
    else:
        # Final evaluation based on score
        if score >= 6:
            reply = f"🎉 Interview completed! Excellent performance. Your score: {score}/10. You are SELECTED!"
        else:
            reply = f"🏁 Interview completed. Thank you for participating. Your score: {score}/10. Unfortunately, you are NOT SELECTED."

    history.append({"role": "assistant", "content": reply})
    return history, score

with gr.Blocks() as demo:
    gr.Markdown("# 🤖 AI HR Interview Chatbot with Scoring")
    gr.Markdown("Answer each question fully. Providing descriptive answers (over 5 words) earns you points!")

    # Initialize state for keeping track of the score
    score_state = gr.State(0)

    chatbot = gr.Chatbot(
        value=[{
            "role": "assistant",
            "content": questions[0]
        }],
        label="Interview"
    )

    msg = gr.Textbox(
        placeholder="Type your answer here...",
        label="Your Answer"
    )

    send = gr.Button("Send")

    send.click(
        respond,
        inputs=[msg, chatbot, score_state],
        outputs=[chatbot, score_state]
    ).then(
        lambda: "",
        outputs=msg
    )

    msg.submit(
        respond,
        inputs=[msg, chatbot, score_state],
        outputs=[chatbot, score_state]
    ).then(
        lambda: "",
        outputs=msg
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0b02e27a8c661ffaf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:

!pip -q install -U gradio google-genai

import gradio as gr
from google import genai
import getpass
import re

# 1. Connect to Gemini
API_KEY = getpass.getpass("Enter your Gemini API key: ").strip()
client = genai.Client(api_key=API_KEY)

# 2. Interview questions
questions = [
    "Tell me about yourself.",
    "Why do you want to work for our company?",
    "What are your strengths?",
    "What are your weaknesses?",
    "Where do you see yourself in 5 years?",
    "Why should we hire you?",
    "Tell me about a challenge you faced and how you solved it.",
    "How do you handle pressure or stress?",
    "What motivates you to do your best work?",
    "Do you have any questions for us?"
]

# 3. AI evaluation
def evaluate_answer(question, answer):
    prompt = f"""
You are a friendly HR interview coach helping a college student.

Interview question: {question}
Student's answer: {answer}

Evaluate only the answer provided. Be fair and encouraging.
Do not assume facts the student has not mentioned.

Give feedback in this format:

SCORE: [integer from 1 to 10]

WHAT YOU DID WELL:
Mention specific positive points.

WHAT TO IMPROVE:
Give practical suggestions.

SAMPLE BETTER ANSWER:
Provide a natural example answer that the student can adapt.
Do not invent personal experiences for the student.
"""
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    feedback = response.text or "No feedback was returned."
    match = re.search(r"SCORE:\s*(\d+)", feedback, re.IGNORECASE)
    score = max(1, min(10, int(match.group(1)))) if match else None

    return feedback, score

# 4. Handle each answer
def chat(message, history, index, scores):
    history = list(history or [])
    scores = list(scores or [])

    if not message.strip():
        return history, "", index, scores, ""

    if index >= len(questions):
        history.append({
            "role": "user",
            "content": message
        })
        history.append({
            "role": "assistant",
            "content": "The interview is already complete. Please start a new interview."
        })
        return history, "", index, scores, ""

    question = questions[index]

    history.append({"role": "user", "content": message})

    try:
        feedback, score = evaluate_answer(question, message)

        if score is not None:
            scores.append(score)

        history.append({
            "role": "assistant",
            "content": f"### Feedback for Question {index + 1}\n\n{feedback}"
        })

    except Exception as e:
        history.append({
            "role": "assistant",
            "content": (
                "I couldn't generate AI feedback. Please check your Gemini "
                "API key, internet connection, and API quota, then try again.\n\n"
                f"Error: {str(e)}"
            )
        })
        return history, "", index, scores, ""

    index += 1
    report = ""

    if index < len(questions):
        history.append({
            "role": "assistant",
            "content": (
                f"### Question {index + 1} of {len(questions)}\n\n"
                f"{questions[index]}"
            )
        })
    else:
        report = "## Final Interview Performance Report\n\n"
        report += f"Questions completed: {index}/{len(questions)}\n\n"

        if scores:
            average = sum(scores) / len(scores)
            report += f"**Average score: {average:.1f}/10**\n\n"
            report += "| Question | Score |\n|---|---:|\n"
            for i, score in enumerate(scores, start=1):
                report += f"| Question {i} | {score}/10 |\n"

        report += (
            "\n### Next steps\n"
            "- Review the feedback for each answer.\n"
            "- Practise adding relevant examples.\n"
            "- Keep answers clear, specific, and honest.\n\n"
            "*Scores are AI-generated practice estimates, not an official hiring assessment.*"
        )

        history.append({
            "role": "assistant",
            "content": "🎉 You have completed all 10 interview questions! Your final report is displayed below."
        })

    return history, "", index, scores, report

# 5. Start a new interview
def start_interview():
    return (
        [{"role": "assistant", "content":
          f"Welcome to the AI HR Interview!\n\n"
          f"**Question 1 of {len(questions)}:**\n\n{questions[0]}"}],
        0,
        [],
        ""
    )

# 6. Build the Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("# 🤖 AI-Powered HR Interview Chatbot")
    gr.Markdown(
        "Answer 10 HR interview questions and receive personalized "
        "AI feedback after each answer."
    )

    chatbot = gr.Chatbot(
        value=[{
            "role": "assistant",
            "content": f"Welcome! **Question 1:**\n\n{questions[0]}"
        }],
        label="Interview"
    )

    index_state = gr.State(0)
    scores_state = gr.State([])

    answer_box = gr.Textbox(
        label="Your Answer",
        placeholder="Type your interview answer here...",
        lines=3
    )

    with gr.Row():
        send_button = gr.Button("Submit Answer", variant="primary")
        restart_button = gr.Button("Restart Interview")

    report_box = gr.Markdown(label="Final Report")

    send_button.click(
        chat,
        inputs=[answer_box, chatbot, index_state, scores_state],
        outputs=[chatbot, answer_box, index_state, scores_state, report_box]
    )

    answer_box.submit(
        chat,
        inputs=[answer_box, chatbot, index_state, scores_state],
        outputs=[chatbot, answer_box, index_state, scores_state, report_box]
    )

    restart_button.click(
        start_interview,
        outputs=[chatbot, index_state, scores_state, report_box]
    ).then(lambda: "", outputs=answer_box)

demo.launch(share=True)

Enter your Gemini API key: ··········
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba95eab4703b0216e1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
